# The LLAMA of WallStreet
## LLM-based Reddit Stock Sentiment Analysis

**Pipeline overview:**
1. Load Reddit comments
2. Use an LLM to extract stock tickers and sentiment from each comment
3. Aggregate sentiment by ticker and date
4. Visualize sentiment trends
5. Design an agent that can run this pipeline via natural language

## 1. Imports & Configuration

Sets up all libraries and connects to the LLM server running on the Leonardo compute node.
Keep `LIMIT` small (50) while testing — increase to 1000+ for the final run.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from enum import Enum
import glob, re, os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from langchain_openai import ChatOpenAI

t0 = datetime.now()

INPUT_FILE  = "reddit_comments.csv"
MODEL_NAME  = "mistralai/Mistral-Small-3.2-24B-Instruct-2506"
API_KEY     = "password"
LIMIT       = 50    # keep small while testing
N_WORKERS   = 8     # parallel LLM calls

# Auto-detect the vLLM node IP from the most recent LLM SLURM job output.
# The job prints: [INFO]: Starting head node lrdn1234 at 10.x.x.x.
def _get_llm_endpoint(search_dir=".", port=8000, fallback="http://10.1.1.53:8000/v1"):
    out_files = sorted(
        glob.glob(os.path.join(search_dir, "llm_launcher_small-*.out")),
        key=os.path.getmtime, reverse=True
    )
    for f in out_files:
        try:
            with open(f) as fp:
                for line in fp:
                    m = re.search(r'Starting head node \S+ at (\d+\.\d+\.\d+\.\d+)', line)
                    if m:
                        ip = m.group(1)
                        print(f"[auto] LLM node IP: {ip}  (from {os.path.basename(f)})")
                        return f"http://{ip}:{port}/v1"
        except Exception:
            continue
    print(f"[auto] No LLM job output found — using fallback: {fallback}")
    return fallback

VLLM_ENDPOINT = _get_llm_endpoint()

llm = ChatOpenAI(base_url=VLLM_ENDPOINT, api_key=API_KEY,
                 model=MODEL_NAME, temperature=0)
print(f"LLM ready  ({VLLM_ENDPOINT})")


## 2. Output Schemas

We use Pydantic to define the exact structure the LLM must return for each comment:
- `tickers`: list of stock ticker symbols found (e.g. `["AAPL", "TSLA"]`)
- `sentiment`: one of 5 levels from *very positive* to *very negative*
- `is_relevant`: `True` only if the comment is about a publicly traded company

`SENTIMENT_MAP` converts text labels to numbers so we can do maths on them.

In [ ]:
class Sentiment(str, Enum):
    VERY_POSITIVE = "very positive"
    POSITIVE      = "positive"
    NEUTRAL       = "neutral"
    NEGATIVE      = "negative"
    VERY_NEGATIVE = "very negative"

class CommentAnalysis(BaseModel):
    tickers:     list[str]  # e.g. ["AAPL", "TSLA"]; empty if not finance-related
    sentiment:   Sentiment
    is_relevant: bool       # True only if a publicly traded company is mentioned

SENTIMENT_MAP = {
    "very positive":  2,
    "positive":       1,
    "neutral":        0,
    "negative":      -1,
    "very negative": -2,
}
print("Schemas defined")

## 3. System Prompt

This is the instruction given to the LLM before every comment. It defines the task,
the output format, and the rules to follow.

> **Teammate 1 — this cell is yours to improve.**  
> Try adding few-shot examples, handling edge cases (e.g. "the Fed raised rates" → not relevant),
> or rephrasing the rules. Re-run Cell 5 after each change to see the effect.

In [ ]:
SYSTEM_PROMPT = """You are a financial NLP system that analyzes Reddit comments for stock market signals. Your goal is to identify stocks mentioned and assess investor-relevant sentiment — not general emotional tone.

CRITICAL: Assess sentiment TOWARD the specific publicly traded company, not the general emotional tone of the comment. A politically-framed or social comment that accuses a company of wrongdoing (data selling, corruption, propaganda, safety failures) is NEGATIVE for that company's stock even if no explicit financial claim is made.

Respond with ONLY a valid JSON object in this exact format:
{"tickers": ["AAPL", "TSLA"], "sentiment": "negative", "is_relevant": true}

## Relevance
- is_relevant: true ONLY if at least one publicly traded company is clearly mentioned or implied
- If not relevant: {"tickers": [], "sentiment": "neutral", "is_relevant": false}
- Comments about specific companies found in any context (news articles, Reddit threads) are relevant
- Only mark is_relevant: false for content with NO company mention (pure politics, sports, personal stories)
- If the comment references "the company" or "they" without naming a company, and you cannot identify the ticker from context, mark is_relevant: false

## Ticker extraction
- Use standard US ticker symbols (e.g. CMG for Chipotle, TSLA for Tesla)
- Include ALL tickers mentioned if multiple companies are discussed
- If a non-traded competitor is mentioned favorably over a public stock, include the public stock ticker as negative
- Subsidiary brands: Taco Bell/KFC/Pizza Hut → YUM; Instagram/WhatsApp → META; YouTube → GOOGL; AWS → AMZN

## Sentiment scale — stock-signal severity
Rate sentiment based on how much this would shift an investor's view of the SPECIFIC COMPANY:

"very negative" — severe reputational or existential damage. Use this for ANY of:
  - Formal violations: fraud, NLRB/SEC violations, food safety crisis, union-busting, executive misconduct
  - Calls for extreme action: explicit boycott, delisting from exchanges, nationalization, government takeover
  - Profanity or strong condemnation directed at the company combined with language expressing the company has no future or the user has permanently severed the relationship
  - Social media or colloquial language that means the company should quit, is finished, or has permanently failed — evaluate the intent behind the expression, not the literal words
  - Accusation that the company is committing financial fraud or theft against customers
  - Permanent customer departure combined with moral condemnation of the company's character or ethics
  - Language that dismisses the company's entire product or service as worthless — strong contempt for the brand itself, even framed as personal opinion
  - Multiple stacked financial catastrophe signals: two or more of the following co-occurring about the same company: sustained stock price decline, major debt burden or cash flow crisis, imminent bankruptcy or insolvency risk, complete failure of a key business segment, loss of a major market

"negative" — real but recoverable complaints: sustained price gouging, product quality failures, employee mistreatment, competitor clearly recommended over this stock, user reframes positive corporate news as predatory without explicit departure, political or ethical accusation naming a specific company (data privacy violations, propaganda, safety negligence, labor exploitation)
  - Ironic or sarcastic reaction to a reported stock purchase or large position, where the tone signals the commenter expects the investment to fail → negative for that stock

"neutral" — minor or ambiguous: trivial product gripes (packaging, one bad experience), political or macro commentary without direct stock impact, mixed signals with no clear direction

"positive" — favorable: share buybacks, strong earnings, first-hand positive customer experience, analyst upgrades

"very positive" — exceptional: blowout earnings, major acquisition wins, transformative positive news

## Classification rules
- Minor operational complaints (poorly wrapped food, slightly small portion, one bad visit) → neutral, NOT negative
- User's explicit first-hand positive experience overrides broader negative narrative → positive
- Employee posts about forced anti-union training or corporate brainwashing → very negative
- NLRB violations combined with angry or cursing language toward executives → very negative
- User recommends a non-traded competitor over a specific public stock → negative for that public stock
- A user stating they have stopped or will stop patronizing a company → negative; if combined with profanity, moral condemnation, or language implying the company has no future → very negative
- When a comment quotes a news article and then adds editorial text, label the sentiment of the user's editorial — not the article
- Output JSON only — no explanation, no markdown, no extra text

## Examples

Comment: "No shit. I stopped going to Chipotle years ago. Fuck this company. They can go out of business for all I care."
Output: {"tickers": ["CMG"], "sentiment": "very negative", "is_relevant": true}

Comment: "Boeing has spent nearly $70B on stock buybacks since 2010. Ban stock buybacks, nationalize the company as a critical security asset."
Output: {"tickers": ["BA"], "sentiment": "very negative", "is_relevant": true}

Comment: "Amazon warehouse workers are collapsing on the floor and management just tells them to keep going. This company needs to be shut down."
Output: {"tickers": ["AMZN"], "sentiment": "very negative", "is_relevant": true}

Comment: "I'd like to know how much data Meta and Google sell to China that America doesn't seem to care about."
Output: {"tickers": ["META", "GOOGL"], "sentiment": "negative", "is_relevant": true}

Comment: "I don't think I've ever thought of Chipotle's portions as generous, they always skimped compared to Qdoba."
Output: {"tickers": ["CMG"], "sentiment": "negative", "is_relevant": true}

Comment: "Funny how the guy comes from Taco Bell via Chipotle to Starbucks, which all three happen to be companies I think sell shit."
Output: {"tickers": ["YUM", "CMG", "SBUX"], "sentiment": "very negative", "is_relevant": true}

Comment: "Oh no…right after the r/wallstreetbets post: 'I just bought 700k worth of Intel Stock'"
Output: {"tickers": ["INTC"], "sentiment": "negative", "is_relevant": true}
"""

structured_llm = llm.with_structured_output(CommentAnalysis)
print("System prompt ready (v5)")

## 4. Single Comment Function & Test

This function sends one comment to the LLM and returns the structured result.
The `try/except` means a single failed request won't crash the whole pipeline.

We test it on a known comment first to confirm the LLM is working correctly.

In [ ]:
def analyze_comment(comment: str) -> dict:
    try:
        result = structured_llm.invoke([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": str(comment)[:2000]},
        ])
        return {
            "tickers":     [t.upper().strip() for t in result.tickers if t.strip()],
            "sentiment":   result.sentiment.value,
            "is_relevant": result.is_relevant,
            "error":       None,
        }
    except Exception as e:
        return {"tickers": [], "sentiment": "neutral", "is_relevant": False, "error": str(e)}

# Sanity check — should return AAPL with positive sentiment
test = analyze_comment("I'm very bullish on Apple, AAPL is going to crush earnings this quarter")
print(test)

## 5. Parallel Pipeline

Processing 100k comments one by one would take hours. Instead we use a `ThreadPoolExecutor`
to send multiple requests to the LLM simultaneously.

Since each call is just waiting for a network response (IO-bound), running 8 in parallel
gives roughly 8x the throughput with no extra hardware.

In [ ]:
def run_pipeline(df: pd.DataFrame, n_workers: int = N_WORKERS) -> list:
    results  = [None] * len(df)
    comments = df["comments"].tolist()
    print(f"Processing {len(comments)} comments with {n_workers} workers...")

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(analyze_comment, c): i
                   for i, c in enumerate(comments)}
        done = 0
        for future in as_completed(futures):
            results[futures[future]] = future.result()
            done += 1
            if done % 10 == 0:
                print(f"  {done}/{len(comments)} done", flush=True)

    errors = sum(1 for r in results if r and r["error"])
    print(f"Finished — {errors} errors out of {len(results)}")
    return results

## 6. Load Data & Run Pipeline

Load the Reddit comments CSV, sample `LIMIT` rows, and run the full pipeline.
With `LIMIT=50` and 8 workers this takes about 1-2 minutes.
Once results look good, go back to Cell 1 and increase `LIMIT`.

In [ ]:
df = pd.read_csv(INPUT_FILE)
df["datetime"] = pd.to_datetime(df["datetime"])
print(f"Full dataset: {len(df):,} rows")
print(f"Date range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")

df_sample  = df.sample(n=LIMIT, random_state=42).reset_index(drop=True)
raw_results = run_pipeline(df_sample)

## 7. Build Results DataFrame

Flatten raw results into a tidy DataFrame — one row per ticker per comment.
A comment mentioning 3 tickers produces 3 rows. Irrelevant comments are dropped.

In [ ]:
rows = []
for i, (_, row) in enumerate(df_sample.iterrows()):
    r = raw_results[i]
    if r and r["is_relevant"] and r["tickers"]:
        for ticker in r["tickers"]:
            rows.append({
                "datetime":       row["datetime"],
                "date":           row["datetime"].date(),
                "ticker":         ticker,
                "sentiment_label": r["sentiment"],
                "sentiment_score": SENTIMENT_MAP.get(r["sentiment"], 0),
                "subreddit":      row["subreddits"],
            })

if not rows:
    errors = [r.get("error") for r in raw_results if r and r.get("error")]
    sample_err = errors[0] if errors else "all results were irrelevant or empty"
    raise RuntimeError(
        f"No relevant ticker-comment pairs found.\n"
        f"LLM endpoint: {VLLM_ENDPOINT}\n"
        f"Sample error: {sample_err}\n"
        "Check the test in Cell 8 — if it fails, the endpoint IP is wrong."
    )

df_results = pd.DataFrame(rows)
print(f"Extracted {len(df_results):,} ticker-comment pairs")
print(f"Unique tickers: {df_results['ticker'].nunique()}")
print("\nTop 10 tickers:")
print(df_results["ticker"].value_counts().head(10))


## 8. Aggregation & Metrics

Compute daily sentiment statistics per ticker, and an overall summary across all dates.

> **Teammate 3 — this cell is yours to expand.**  
> Ideas: sentiment by subreddit, most volatile tickers (high std dev),
> day-over-day sentiment change, tickers with the most negative trend.

In [ ]:
daily = (
    df_results.groupby(["date", "ticker"])["sentiment_score"]
    .agg(avg_sentiment="mean", min_sentiment="min",
         max_sentiment="max",  n_comments="count")
    .reset_index()
)
daily["date"] = pd.to_datetime(daily["date"])

ticker_summary = (
    df_results.groupby("ticker")["sentiment_score"]
    .agg(total_mentions="count", avg_sentiment="mean",
         min_sentiment="min",    max_sentiment="max",
         sentiment_std="std")
    .sort_values("total_mentions", ascending=False)
    .reset_index()
)

print("Ticker summary (top 10):")
print(ticker_summary.head(10).to_string(index=False))

## 9. Visualization

Plot daily average sentiment for the top tickers with a trend line.

> **Teammate 2 — this cell is yours to improve.**  
> Ideas: bar chart of top tickers by mention count, heatmap of sentiment by subreddit,
> annotations on big sentiment spikes, side-by-side comparison of multiple tickers.

In [ ]:
try:
    import yfinance as yf
    _YFINANCE = True
except ImportError:
    _YFINANCE = False
    print("yfinance not installed — price overlay disabled. Run: pip install yfinance")

def plot_ticker(ticker: str):
    data = daily[daily["ticker"] == ticker].sort_values("date")
    if len(data) < 2:
        print(f"Not enough data for {ticker}")
        return

    fig, ax1 = plt.subplots(figsize=(12, 5))

    # Sentiment (left axis)
    ax1.plot(data["date"], data["avg_sentiment"],
             marker="o", color="steelblue", linewidth=1.5, label="Avg sentiment", zorder=3)
    x = np.arange(len(data))
    z = np.polyfit(x, data["avg_sentiment"], 1)
    ax1.plot(data["date"], np.poly1d(z)(x),
             "--", color="orange", linewidth=1.5, label="Trend")
    ax1.axhline( 1, color="green", alpha=0.2)
    ax1.axhline( 0, color="gray",  alpha=0.2)
    ax1.axhline(-1, color="red",   alpha=0.2)
    ax1.set_ylabel("Sentiment (-2 to +2)", color="steelblue")
    ax1.set_ylim(-2.5, 2.5)
    ax1.tick_params(axis="y", labelcolor="steelblue")

    # Price overlay (right axis)
    if _YFINANCE:
        try:
            start = pd.Timestamp(data["date"].min()) - pd.Timedelta(days=1)
            end   = pd.Timestamp(data["date"].max()) + pd.Timedelta(days=2)
            prices = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if not prices.empty:
                ax2 = ax1.twinx()
                ax2.plot(prices.index, prices["Close"],
                         color="black", linewidth=1.2, alpha=0.55, label=f"{ticker} price")
                ax2.set_ylabel("Stock price ($)", color="black")
                ax2.tick_params(axis="y", labelcolor="black")
                lines2, labels2 = ax2.get_legend_handles_labels()
            else:
                lines2, labels2 = [], []
        except Exception as e:
            print(f"Price fetch failed for {ticker}: {e}")
            lines2, labels2 = [], []
    else:
        lines2, labels2 = [], []

    lines1, labels1 = ax1.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")
    ax1.set_title(f"{ticker} — Reddit Sentiment vs Stock Price", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

top_tickers = ticker_summary.head(5)["ticker"].tolist()
for t in top_tickers:
    plot_ticker(t)

## 10. Agent Design

Here we describe the agentic system that lets a non-technical user trigger this pipeline
via natural language through a chatbot. The agent translates plain English into SLURM commands.

### Tools

| Tool | What it does |
|------|--------------|
| `submit_job(n_comments, output_dir)` | Writes and submits `student_job_LLM.py` as a SLURM batch job via `sbatch` |
| `get_job_status(job_id)` | Checks job progress via `sacct` |
| `list_my_jobs()` | Lists all active/queued jobs via `squeue` |
| `read_results(output_dir)` | Reads completed CSVs and returns a plain-text summary |
| `cancel_job(job_id)` | Cancels a running job via `scancel` |

### System Prompt

> You are LeonardoOps, an HPC job manager for CINECA's Leonardo supercomputer.  
> You help data scientists run the Reddit sentiment pipeline using natural language.  
> Always confirm parameters before submitting a job.  
> Translate SLURM status codes (PD, R, CG, FAILED) into plain English.  
> When results are ready, summarize the top 5 tickers and their average sentiment.  
> Never cancel a job without explicit user confirmation.

## 11. Agent Implementation (Optional)

Implement the agent described above using LangChain and the same LLM.
This is the optional bonus task — completing it demonstrates the full end-to-end vision.

In [ ]:
import subprocess
from pathlib import Path
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

AGENT_SYSTEM_PROMPT = """You are LeonardoOps, an HPC job manager for CINECA's Leonardo supercomputer.
You help data scientists run the Reddit sentiment pipeline using natural language.
Always confirm parameters before submitting a job.
Translate SLURM status codes (PD, R, CG, FAILED) into plain English.
When results are ready, summarize the top 5 tickers and their average sentiment.
Never cancel a job without explicit user confirmation.
"""

@tool
def submit_job(n_comments: int = 10000, output_dir: str = "./output") -> str:
    """Submit the sentiment pipeline as a SLURM batch job."""
    script = f"""#!/bin/bash
#SBATCH --job-name=sentiment_pipeline
#SBATCH --account=tra26_bbs
#SBATCH --partition=boost_usr_prod
#SBATCH --nodes=1 --cpus-per-task=32 --gres=gpu:4
#SBATCH --time=04:00:00
#SBATCH --output=sentiment-%j.out
#SBATCH --error=sentiment-%j.err

source /leonardo_scratch/fast/tra26_bbs/intro_to_llms_venv/bin/activate
python run_inference.py --output {output_dir}
"""
    Path("config/sentiment_job.sh").write_text(script)
    result = subprocess.run(["sbatch", "config/sentiment_job.sh"],
                            capture_output=True, text=True)
    return result.stdout or result.stderr

@tool
def get_job_status(job_id: int) -> str:
    """Get the status of a SLURM job by ID."""
    result = subprocess.run(
        ["sacct", "-j", str(job_id),
         "--format=JobID,State,Elapsed,NodeList", "-P"],
        capture_output=True, text=True)
    return result.stdout or "No output — job may not exist yet."

@tool
def list_my_jobs() -> str:
    """List all jobs currently queued or running for this user."""
    result = subprocess.run(["squeue", "--me"], capture_output=True, text=True)
    return result.stdout or "No active jobs."

@tool
def read_results(output_dir: str = "./output") -> str:
    """Read completed results and return a plain-text summary of top tickers."""
    p = Path(output_dir) / "reddit_sentiment.csv"
    if not p.exists():
        return f"No results found at {p}. Has the job finished?"
    df = pd.read_csv(p)
    relevant = df[df["is_relevant"] == True]
    if relevant.empty:
        return "No relevant comments found in results."
    summary = (relevant.groupby("tickers")["sentiment"]
               .value_counts().reset_index(name="count")
               .head(10))
    return f"Top results:\n{summary.to_string(index=False)}"

@tool
def cancel_job(job_id: int) -> str:
    """Cancel a running or queued SLURM job."""
    result = subprocess.run(["scancel", str(job_id)],
                            capture_output=True, text=True)
    return f"Job {job_id} cancelled." if result.returncode == 0 else result.stderr

agent = create_react_agent(
    llm,
    tools=[submit_job, get_job_status, list_my_jobs, read_results, cancel_job],
    checkpointer=MemorySaver(),
    state_modifier=SystemMessage(content=AGENT_SYSTEM_PROMPT),
)

def run_agent(message: str, thread_id: str = "agent-1"):
    config = {"configurable": {"thread_id": thread_id}}
    for state in agent.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config,
        stream_mode="values",
    ):
        last = state["messages"][-1]
        if isinstance(last, AIMessage) and last.content:
            print(f"[Agent]: {last.content}")
        elif isinstance(last, ToolMessage):
            print(f"[Tool - {last.name}]: {last.content[:300]}")

# Test the agent
run_agent("Do I have any jobs currently running on Leonardo?")

## Timing

In [ ]:
print(f"Total execution time: {datetime.now() - t0}")